# HW02-1：股票收益与组合风险分析

- **姓名：**朱家瑞
- **学号：**待填写（坚果云提交前补充完整学号）
- **作业简介或教师作业页面链接：**https://lianxhcn.github.io/FinEco/exercises/hw-02.html
- **个人 GitHub 仓库访问地址：**https://github.com/Jaremy112/fineco-homework
- **HW02-1 目录链接：**https://github.com/Jaremy112/fineco-homework/tree/main/hw-02
- **HW02-2 目录链接：**待完成HW02-2后补充
- **数据来源、获取日期与样本期间：**AKShare 1.18.39（新浪财经日行情、巨潮资讯股本变动）为主要数据源，CSMAR上市公司基本信息年度表用于统一简称、上市日期及2025年末行业分类，Wind金融数据服务用于关键字段交叉核验；获取日期为2026-09-18；正式样本为2021-01-01至2026-09-16，并补充每只股票在2021年前最后一个有效交易日用于计算首个日收益率。
- **AI 使用声明：**使用了Hermes Agent（Codex）辅助读取教师作业要求、设计数据获取与审计流程、编写和检查Python代码；使用Wind MCP调用万得金融数据服务。股票池、字段口径、异常处理和结论由本人核对，Notebook保留数据来源、参数、实际输出和局限说明。

## 分析目的与总体思路

本文选择10只至少覆盖5个CSMAR行业、且均在样本开始前上市的A股，先取得不复权收盘价、经公司行为调整的复权收盘价、成交量、成交额和历史总市值，再检查主键、日期、覆盖、缺失、停牌和异常收益。在统一的数据口径上，依次完成价格与累计收益图、日收益及滚动波动率、描述统计、相关性分析，并比较每日等权与使用上一交易日总市值确定权重的市值加权组合。分析遵循“说明—代码与实际输出—结果解读”的结构。

## 1. 选股与获取数据

### Step 1：Markdown分析说明

本节要回答两个问题：第一，所选10只股票是否满足至少覆盖5个行业、在2021年以前上市且不依据样本期事后涨幅选取；第二，取得的数据是否覆盖作业要求的价格、收益率、流动性和历史总市值口径。先冻结股票池再下载，可以避免根据2021—2026年的实际表现替换股票而产生事后选择。

本作业优先使用AKShare获取行情：不复权和后复权日行情来自新浪财经接口，股本变动来自巨潮资讯接口。由于本题要求历史**总市值**，而新浪行情接口只直接提供流通股本，本文用巨潮资讯按变动日期记录的总股本与同日不复权收盘价构造历史总市值；不使用流通市值，也不使用复权价格乘股本。Wind用于股票档案、统一行业分类和关键日期的总市值交叉核验，以减少代码、日期、单位和口径误判。复权价格仅作为连续收益分析尺度，不视为历史实际成交价。

分析要点和口径如下：

- 股票池：10只A股，统一采用CSMAR上市公司基本信息年度表中的2025年末行业分类，数据读取日期为2026-09-18；
- 选股依据：行业、商业模式和代表性，不以样本期事后涨幅为依据；
- 样本：2021-01-01至2026-09-16，并补充2020年最后一个有效交易日；
- 价格：新浪财经不复权收盘价用于价格时序图；新浪财经后复权收盘价用于日简单收益率；
- 流动性：新浪财经成交量原始单位为股；成交额由元转换为亿元；
- 市值：巨潮资讯总股本原始单位为万股，按变动日期向后匹配；历史总市值（亿元）=`不复权收盘价×总股本（万股）/10000`；
- 组合权重：交易日 $t$ 的市值权重只使用 $t-1$ 日历史总市值；
- 获取审计：保留原始CSV、接口参数、字段单位、覆盖、SHA-256和处理后数据；
- 交叉核验：以贵州茅台2020-12-31为例，将构造总市值与Wind同日历史总市值比较。

### 选股表

| 代码 | 简称 | 行业 | 上市日期 | 选股理由 |
|---|---|---|---|---|
| 600519.SH | 贵州茅台 | 酒、饮料和精制茶制造业 | 2001-08-27 | 消费品牌代表，覆盖品牌型商业模式 |
| 600036.SH | 招商银行 | 货币金融服务 | 2002-04-09 | 银行业代表，覆盖金融中介业务 |
| 600276.SH | 恒瑞医药 | 医药制造业 | 2000-10-18 | 医药制造代表，覆盖研发驱动模式 |
| 000333.SZ | 美的集团 | 电气机械和器材制造业 | 2013-09-18 | 家电制造代表，覆盖耐用消费与制造 |
| 002594.SZ | 比亚迪 | 汽车制造业 | 2011-06-30 | 汽车制造代表，覆盖技术投入与产业链 |
| 600309.SH | 万华化学 | 化学原料和化学制品制造业 | 2001-01-05 | 化工材料代表，覆盖周期制造与研发投入 |
| 002352.SZ | 顺丰控股 | 邮政业 | 2010-02-05 | 物流服务代表，覆盖网络型服务商业模式 |
| 600030.SH | 中信证券 | 资本市场服务 | 2003-01-06 | 证券业代表，补充非银行金融模式 |
| 601012.SH | 隆基绿能 | 电气机械和器材制造业 | 2012-04-11 | 新能源设备代表，覆盖制造周期 |
| 002475.SZ | 立讯精密 | 计算机、通信和其他电子设备制造业 | 2010-09-15 | 电子制造代表，覆盖消费电子供应链 |

上述10只股票覆盖9个CSMAR行业，全部在2021-01-01之前上市。股票池不是依据2021—2026年已实现涨幅事后筛选，而是依据行业、商业模式和代表性事先确定。

### 数据来源与接口说明

本题的数据源可以使用，但实际获取采用分工明确的组合，而不是把所有数据强行交给单一数据库：

- **AKShare 1.18.39 / 新浪财经 `stock_zh_a_daily`**：获取不复权日收盘价、后复权日收盘价、成交量和成交额；
- **AKShare 1.18.39 / 巨潮资讯 `stock_share_change_cninfo`**：获取总股本及股本变动日期，用于构造历史总市值；
- **CSMAR上市公司基本信息年度表**：统一获取简称、上市日期和2025年末行业分类；
- **Wind金融数据服务**：用于股票档案和贵州茅台关键日期总市值交叉核验。Wind全量行情调用因账户积分余额不足未采用，不影响本题使用AKShare完成数据获取。

因此，本题并不是“AKShare或CSMAR不能用”，而是：AKShare负责行情和股本，CSMAR负责统一选股信息，Wind负责交叉核验。所有原始返回和处理后数据均已保存到本Notebook同目录的`data/raw/akshare/`和`data/processed/`。


### Step 2：代码与实际输出

下面先显示本次运行环境，再展示冻结后的选股表、数据来源清单、各股票覆盖情况、主键检查和历史总市值交叉核验。

In [1]:
# 导入本节所需包并设置显示选项
from pathlib import Path
import json
import platform
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd()
if not (PROJECT / "audit" / "source_manifest_akshare.csv").exists():
    # 允许从个人仓库根目录启动Notebook
    candidate = Path.cwd() / "hw-02"
    if (candidate / "audit" / "source_manifest_akshare.csv").exists():
        PROJECT = candidate
    else:
        raise FileNotFoundError("未找到HW02-1审计文件，请先运行数据下载脚本。")

pd.set_option("display.max_colwidth", 60)
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("项目目录:", PROJECT.resolve())

Python: 3.12.12
pandas: 2.3.3
项目目录: /Users/macbookairzhu/Documents/fineco-homework/hw-02


In [2]:
# 从唯一股票池主表读取10只股票及统一的CSMAR 2025年末行业分类
selection_master_path = PROJECT / "data" / "processed" / "stock_selection_master.csv"
selection_master = pd.read_csv(
    selection_master_path,
    dtype={"stock_code": "string", "symbol": "string"},
    parse_dates=["listing_date", "industry_classification_date", "industry_source_read_date", "sample_start", "sample_end"],
)

required_columns = [
    "stock_code", "stock_name", "industry", "listing_date", "selection_reason",
    "industry_standard", "industry_classification_date", "industry_source",
    "industry_source_read_date", "listed_before_sample",
]
missing_columns = sorted(set(required_columns) - set(selection_master.columns))
if missing_columns:
    raise ValueError(f"股票池主表缺少字段: {missing_columns}")
if len(selection_master) != 10 or selection_master["stock_code"].duplicated().any():
    raise ValueError("股票池主表必须恰好包含10只不重复股票")

selection = selection_master[[
    "stock_code", "stock_name", "industry", "listing_date", "selection_reason",
    "listed_before_sample", "industry_standard", "industry_classification_date",
    "industry_source", "industry_source_read_date",
]].rename(columns={
    "stock_code": "股票代码",
    "stock_name": "股票简称",
    "industry": "CSMAR行业（2025年末）",
    "listing_date": "上市日期",
    "selection_reason": "选股理由",
    "listed_before_sample": "2021年前上市",
    "industry_standard": "行业分类标准",
    "industry_classification_date": "行业分类日期",
    "industry_source": "行业分类来源",
    "industry_source_read_date": "来源读取日期",
})

display(selection)
print("股票池主表:", selection_master_path.resolve())
print("股票数:", len(selection))
print("CSMAR行业数量:", selection["CSMAR行业（2025年末）"].nunique())
print("全部在样本前上市:", bool(selection["2021年前上市"].all()))

,股票代码,股票简称,CSMAR行业（2025年末）,上市日期,选股理由,2021年前上市,行业分类标准,行业分类日期,行业分类来源,来源读取日期
0,600519.SH,贵州茅台,酒、饮料和精制茶制造业,2001-08-27,消费品牌代表，覆盖品牌型商业模式,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
1,600036.SH,招商银行,货币金融服务,2002-04-09,银行业代表，覆盖金融中介业务,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
2,600276.SH,恒瑞医药,医药制造业,2000-10-18,医药制造代表，覆盖研发驱动模式,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
3,000333.SZ,美的集团,电气机械和器材制造业,2013-09-18,家电制造代表，覆盖耐用消费与制造,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
4,002594.SZ,比亚迪,汽车制造业,2011-06-30,汽车制造代表，覆盖技术投入与产业链,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
5,600309.SH,万华化学,化学原料和化学制品制造业,2001-01-05,化工材料代表，覆盖周期制造与研发投入,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
6,002352.SZ,顺丰控股,邮政业,2010-02-05,物流服务代表，覆盖网络型服务商业模式,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
7,600030.SH,中信证券,资本市场服务,2003-01-06,证券业代表，补充非银行金融模式,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
8,601012.SH,隆基绿能,电气机械和器材制造业,2012-04-11,新能源设备代表，覆盖制造周期,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18
9,002475.SZ,立讯精密,计算机、通信和其他电子设备制造业,2010-09-15,电子制造代表，覆盖消费电子供应链,True,CSMAR上市公司行业分类,2025-12-31,CSMAR上市公司基本信息年度表,2026-09-18


股票池主表: /Users/macbookairzhu/Documents/fineco-homework/hw-02/data/processed/stock_selection_master.csv
股票数: 10
CSMAR行业数量: 9
全部在样本前上市: True


In [3]:
# 读取AKShare下载脚本生成的来源清单、覆盖审计和统一日度数据
manifest = pd.read_csv(PROJECT / "audit" / "source_manifest_akshare.csv")
coverage = pd.read_csv(PROJECT / "audit" / "coverage_akshare.csv")
daily = pd.read_csv(
    PROJECT / "data" / "processed" / "daily_stock_data_20201231_20260916.csv",
    parse_dates=["date", "effective_date"], dtype={"stock_code": "string"}
)
execution = json.loads((PROJECT / "audit" / "execution_akshare.json").read_text(encoding="utf-8"))
crosscheck = json.loads((PROJECT / "audit" / "market_cap_crosscheck.json").read_text(encoding="utf-8"))

display(manifest.groupby("kind").agg(
    文件数=("path", "size"), 原始行数=("rows", "sum")
))
display(coverage)
print("AKShare版本:", execution["akshare_version"])
print("价格上游:", execution["price_upstream"])
print("总股本上游:", execution["share_capital_upstream"])
print("正式样本:", " 至 ".join(execution["sample_period"]))
print("股票-日期主键重复数:", int(daily.duplicated(["stock_code", "date"]).sum()))
print("处理后总行数:", len(daily))
print("总市值交叉核验:")
display(pd.DataFrame([crosscheck]))

,文件数,原始行数
kind,,
cninfo_share_capital,10,815
sina_hfq,10,14061
sina_unadjusted,10,14061


,stock_code,stock_name,industry,rows,date_min,date_max,unadjusted_nonmissing,hfq_nonmissing,market_cap_nonmissing,zero_volume_days
0,000333.SZ,美的集团,电气机械和器材制造业,1385,2020-12-31,2026-09-16,1385,1385,1385,0
1,002352.SZ,顺丰控股,邮政业,1382,2020-12-31,2026-09-16,1382,1382,1382,0
2,002475.SZ,立讯精密,计算机、通信和其他电子设备制造业,1385,2020-12-31,2026-09-16,1385,1385,1385,0
3,002594.SZ,比亚迪,汽车制造业,1385,2020-12-31,2026-09-16,1385,1385,1385,0
4,600030.SH,中信证券,资本市场服务,1379,2020-12-31,2026-09-16,1379,1379,1379,0
5,600036.SH,招商银行,货币金融服务,1385,2020-12-31,2026-09-16,1385,1385,1385,0
6,600276.SH,恒瑞医药,医药制造业,1385,2020-12-31,2026-09-16,1385,1385,1385,0
7,600309.SH,万华化学,化学原料和化学制品制造业,1385,2020-12-31,2026-09-16,1385,1385,1385,0
8,600519.SH,贵州茅台,酒、饮料和精制茶制造业,1385,2020-12-31,2026-09-16,1385,1385,1385,0
9,601012.SH,隆基绿能,电气机械和器材制造业,1385,2020-12-31,2026-09-16,1385,1385,1385,0


AKShare版本: 1.18.39
价格上游: 新浪财经 via AKShare stock_zh_a_daily
总股本上游: 巨潮资讯 via AKShare stock_share_change_cninfo
正式样本: 2021-01-01 至 2026-09-16
股票-日期主键重复数: 0
处理后总行数: 13841
总市值交叉核验:


,stock_code,date,akshare_close_unadjusted_cny,cninfo_total_shares_10k,constructed_market_cap_100m_cny,wind_market_cap_100m_cny_rounded,absolute_difference_100m_cny
0,600519.SH,2020-12-31,1998.0,125619.78,25098.832044,25099.0,0.167956


### Step 3：Markdown结果解读

本节最终保留10只股票，按CSMAR 2025年末行业分类覆盖酒饮料、银行、医药、家电与电气设备、汽车、化工材料、邮政物流、资本市场和电子制造等9个行业，明显高于作业要求的5个行业；10只股票均在2021年以前上市。选股理由围绕行业、商业模式和代表性事先设定，没有使用2021—2026年的已实现涨幅作为筛选条件。与初始候选方案相比，中国神华、中国石化和中国联通被万华化学、顺丰控股和立讯精密替换，以降低大型国有企业的集中度并扩大商业模式差异。

AKShare 1.18.39实际取得13,841条股票—日期记录。10只股票均从2020-12-31覆盖到2026-09-16，其中8只股票各有1,385个交易日；顺丰控股为1,382日，中信证券为1,379日。所有已返回记录的不复权收盘价、后复权收盘价和构造的历史总市值均非缺失，股票—日期主键重复数为0。两只股票观测日较少并不自动等于数据错误，后续清洗将把它们与共同交易日历比较，区分停牌与接口缺口，且不会把缺失收益直接填为0。

历史总市值使用不复权价格和当日有效总股本构造。以贵州茅台2020-12-31为例，不复权收盘价为1,998元，巨潮资讯总股本为125,619.78万股，得到总市值25,098.832044亿元；Wind同日显示值为25,099.0亿元，两者仅相差0.167956亿元，差异与Wind按万亿元四位小数展示的四舍五入相符。这项核验支持总股本单位和构造公式，但不等于所有股票、所有日期都已由第二来源逐日验证。

核心发现：

- 股票池共10只，按统一CSMAR口径覆盖9个行业，且全部在样本开始前上市；
- 处理后共有13,841条记录，主键无重复，价格和总市值核心字段在已有记录中无缺失；
- 顺丰控股和中信证券分别少于共同上限3日和6日，必须在下一节定位日期后再决定处理；
- 贵州茅台的总市值构造值与Wind交叉核验高度一致，但复权价格仍只用于收益计算，不能视为实际成交价格。


## 2. 数据检查与收益率构造

### Step 1：Markdown分析说明

本节要回答四个问题：第一，主键、日期排序和各股票覆盖是否可靠；第二，相比共同交易日历缺失的观测为什么不填零；第三，日收益率用什么口径，停牌日与复牌日分别怎么处理；第四，单位、数量和涨跌幅限制是否与A股实际相符。

上一节已发现顺丰控股和中信证券的观测日少于其余股票，本节先把它与共同交易日历比较，区分停牌与接口缺口，再构造收益率。关键风险是：若直接按组内相邻行计算收益，停牌后复牌当日的收益实际跨越了整个停牌期，把它当作单日收益会夸大波动并污染描述统计与相关性。

分析要点与处理规则如下：

- 主键：检查 `(股票代码, 日期)` 是否唯一，并按代码、日期升序排列；
- 覆盖：以10只股票交易日的并集作为共同交易日历，逐只股票比较缺失日；
- 缺失成因：若某日其余9只股票均有行情而该股无行情，且缺口连续，则为个股停牌，不是接口返回缺口；
- 停牌日：不生成收益率，也不填0，观测直接缺失；
- 收益率：按股票内相邻观测用后复权收盘价计算，$r_{i,t}=P^{hfq}_{i,t}/P^{hfq}_{i,t-1}-1$；
- 跨期收益：若某观测的前一观测不是共同日历上的前一个交易日，则标记为跨期收益，主口径置为缺失，原始值保留在 `ret_raw` 供敏感性比较；
- 单位检查：核对价格、成交量、成交额和总市值的数量级，检查非正价格与零成交量日；
- 合理性核对：10只股票均为主板（000/002/600/601），日涨跌幅限制为±10%，据此判断极端收益是否越界。

### Step 2：代码与实际输出

下面依次展示主键与覆盖检查、单位与数量级检查、缺失日与停牌证据、收益率构造与样本流，以及异常收益与涨跌幅限制核对。

In [4]:
# 主键、排序、覆盖与单位检查
daily = daily.sort_values(["stock_code", "date"]).reset_index(drop=True)

print("股票—日期主键重复数:", int(daily.duplicated(["stock_code", "date"]).sum()))
print("每只股票日期单调递增:", bool(
    daily.groupby("stock_code")["date"].apply(lambda s: s.is_monotonic_increasing).all()
))

# 共同交易日历取正式样本期内10只股票交易日的并集
calendar = sorted(daily.loc[daily["date"] >= "2021-01-01", "date"].unique())
n_sample = int((daily["date"] >= "2021-01-01").sum())
print("共同交易日历交易日数:", len(calendar))
print("正式样本期理论观测数:", len(calendar) * daily["stock_code"].nunique(), " 实际观测数:", n_sample)

# 单位与数量级检查
display(daily[["close_unadjusted", "close_hfq", "volume_shares",
               "turnover_100m_cny", "market_cap_100m_cny"]].describe().T)
print("非正后复权收盘价:", int((daily["close_hfq"] <= 0).sum()),
      " 零成交量交易日:", int((daily["volume_shares"] == 0).sum()))

股票—日期主键重复数: 0
每只股票日期单调递增: True
共同交易日历交易日数: 1384
正式样本期理论观测数: 13840  实际观测数: 13831


,count,mean,std,min,25%,50%,75%,max
close_unadjusted,13841.0,2.299282e+02,4.904733e+02,1.100000e+01,3.505000e+01,5.287000e+01,9.035000e+01,2.601000e+03
close_hfq,13841.0,2.327313e+03,3.924545e+03,1.061600e+02,2.284900e+02,3.797700e+02,2.869400e+03,1.978265e+04
volume_shares,13841.0,5.256221e+07,5.782688e+07,1.032104e+06,1.637536e+07,3.645530e+07,6.882425e+07,1.351455e+09
turnover_100m_cny,13841.0,3.089285e+01,2.679595e+01,1.496820e+00,1.363530e+01,2.324471e+01,3.863727e+01,4.219736e+02
market_cap_100m_cny,13841.0,6.100088e+03,5.714522e+03,8.335876e+02,2.627511e+03,3.688439e+03,7.640314e+03,3.267370e+04


非正后复权收盘价: 0  零成交量交易日: 0


In [5]:
# 缺失交易日与停牌证据（由 scripts/locate_missing_days.py 生成）
missing_days = pd.read_csv(PROJECT / "audit" / "missing_days_akshare.csv")
display(missing_days)
print("缺失观测合计:", len(missing_days))
print("缺失日当日其余9只均有行情的比例:",
      float((missing_days["other_stocks_with_data"] == missing_days["other_stocks_total"]).mean()))

# 公开公告查证结果（来源见解读）
verified = pd.DataFrame([
    {"股票": "顺丰控股 002352.SZ", "停牌区间": "2021-02-05 至 2021-02-09",
     "交易日数": 3, "事由": "筹划收购嘉里物流股权，2月5日开市起临时停牌，2月10日复牌",
     "来源": "证券时报/中证网 2021-02-10；南方都市报 2021-02-05"},
    {"股票": "中信证券 600030.SH", "停牌区间": "2022-01-19 至 2022-01-26",
     "交易日数": 6, "事由": "A股配股缴款期（1/19—1/25）全天停牌，1/26网上清算继续停牌，1/27开市复牌",
     "来源": "中信证券公告 2022-01-13、2022-01-19；中证网 2022-01-13"},
])
display(verified)

,stock_code,stock_name,missing_date,other_stocks_with_data,other_stocks_total,note
0,002352.SZ,顺丰控股,2021-02-05,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
1,002352.SZ,顺丰控股,2021-02-08,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
2,002352.SZ,顺丰控股,2021-02-09,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
3,600030.SH,中信证券,2022-01-19,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
4,600030.SH,中信证券,2022-01-20,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
5,600030.SH,中信证券,2022-01-21,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
6,600030.SH,中信证券,2022-01-24,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
7,600030.SH,中信证券,2022-01-25,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征
8,600030.SH,中信证券,2022-01-26,9,9,其余股票当日均有行情，缺口连续时为个股停牌的典型特征


缺失观测合计: 9
缺失日当日其余9只均有行情的比例: 1.0


,股票,停牌区间,交易日数,事由,来源
0,顺丰控股 002352.SZ,2021-02-05 至 2021-02-09,3,筹划收购嘉里物流股权，2月5日开市起临时停牌，2月10日复牌,证券时报/中证网 2021-02-10；南方都市报 2021-02-05
1,中信证券 600030.SH,2022-01-19 至 2022-01-26,6,A股配股缴款期（1/19—1/25）全天停牌，1/26网上清算继续停牌，1/27开市复牌,中信证券公告 2022-01-13、2022-01-19；中证网 2022-01-13


In [6]:
# 构造日收益率，并标记跨越停牌期的观测
d = daily.copy()
d["ret_raw"] = d.groupby("stock_code")["close_hfq"].pct_change()

# 用共同交易日历的位置差识别跨期：位置差大于1说明中间跳过了交易日
prev_date = d.groupby("stock_code")["date"].shift(1)
cal_pos = {day: i for i, day in enumerate(calendar)}
pos_now, pos_prev = d["date"].map(cal_pos), prev_date.map(cal_pos)
d["skipped_days"] = (pos_now - pos_prev - 1).where(pos_prev.notna() & (pos_now - pos_prev > 1), 0)
d["cross_period"] = pos_prev.notna() & (pos_now - pos_prev > 1)

# 主口径：跨期收益不作为单日收益，置为缺失；原始值保留在 ret_raw
d["ret"] = d["ret_raw"].where(~d["cross_period"])

print("跨期收益明细（跨越交易日数 > 0）:")
display(d.loc[d["cross_period"], ["stock_code", "date", "skipped_days", "close_hfq", "ret_raw", "ret"]])

in_sample = d["date"] >= "2021-01-01"
print("原始日收益数:", int(d.loc[in_sample, "ret_raw"].notna().sum()))
print("其中跨期收益:", int(d["cross_period"].sum()))
print("主口径有效日收益数:", int(d.loc[in_sample, "ret"].notna().sum()))

display(d.loc[in_sample].groupby("stock_code").agg(
    观测数=("date", "size"), 有效收益=("ret", "count"), 跨期剔除=("cross_period", "sum")
))

# 样本流：逐步记录输入、处理与输出
sample_flow = pd.DataFrame([
    {"步骤": "原始记录（含2020年支持日）", "输入": len(d), "输出": len(d), "变化": 0, "原因": "—"},
    {"步骤": "进入正式样本期", "输入": len(d), "输出": int(in_sample.sum()),
     "变化": int(in_sample.sum()) - len(d), "原因": "剔除10条2020-12-31支持日，只用于生成首个收益"},
    {"步骤": "停牌日不生成收益", "输入": len(calendar) * 10, "输出": int(in_sample.sum()),
     "变化": int(in_sample.sum()) - len(calendar) * 10, "原因": "顺丰3日、中信6日停牌，观测缺失且不填0"},
    {"步骤": "剔除跨期收益", "输入": int(in_sample.sum()), "输出": int(d.loc[in_sample, "ret"].notna().sum()),
     "变化": int(d.loc[in_sample, "ret"].notna().sum()) - int(in_sample.sum()),
     "原因": "复牌日收益跨越停牌期，不当作单日收益"},
])
display(sample_flow)

# 保存派生数据供后续各节使用
returns_path = PROJECT / "data" / "processed" / "daily_returns_20210104_20260916.csv"
d.loc[in_sample, ["stock_code", "stock_name", "date", "industry", "close_unadjusted", "close_hfq",
                  "ret", "ret_raw", "cross_period", "skipped_days", "volume_shares",
                  "turnover_100m_cny", "market_cap_100m_cny"]].to_csv(
    returns_path, index=False, encoding="utf-8-sig")
print("已保存派生收益率数据:", returns_path.relative_to(PROJECT))

跨期收益明细（跨越交易日数 > 0）:


,stock_code,date,skipped_days,close_hfq,ret_raw,ret
1410,002352.SZ,2021-02-10,3.0,406.95,0.100043,NaN
5792,600030.SH,2022-01-27,6.0,144.97,-0.003300,NaN


原始日收益数: 13831
其中跨期收益: 2
主口径有效日收益数: 13829


,观测数,有效收益,跨期剔除
stock_code,,,
000333.SZ,1384,1384,0
002352.SZ,1381,1380,1
002475.SZ,1384,1384,0
002594.SZ,1384,1384,0
600030.SH,1378,1377,1
600036.SH,1384,1384,0
600276.SH,1384,1384,0
600309.SH,1384,1384,0
600519.SH,1384,1384,0


,步骤,输入,输出,变化,原因
0,原始记录（含2020年支持日）,13841,13841,0,—
1,进入正式样本期,13841,13831,-10,剔除10条2020-12-31支持日，只用于生成首个收益
2,停牌日不生成收益,13840,13831,-9,顺丰3日、中信6日停牌，观测缺失且不填0
3,剔除跨期收益,13831,13829,-2,复牌日收益跨越停牌期，不当作单日收益


已保存派生收益率数据: data/processed/daily_returns_20210104_20260916.csv


In [7]:
# 异常收益与涨跌幅限制核对
ret = d.loc[d["date"] >= "2021-01-01", ["stock_code", "stock_name", "date", "ret", "ret_raw", "cross_period"]].dropna(subset=["ret"])

print("|r| > 10% 的观测数:", int((ret["ret"].abs() > 0.10).sum()),
      " |r| > 11% 的观测数:", int((ret["ret"].abs() > 0.11).sum()))
print("日收益极值 top 10（主口径）:")
display(ret.reindex(ret["ret"].abs().sort_values(ascending=False).index).head(10))

print("被剔除的跨期收益原始值（保留在 ret_raw，供敏感性比较）:")
display(d.loc[d["cross_period"], ["stock_code", "stock_name", "date", "skipped_days", "ret_raw"]])

|r| > 10% 的观测数: 22  |r| > 11% 的观测数: 0
日收益极值 top 10（主口径）:


,stock_code,stock_name,date,ret,ret_raw,cross_period
13364,601012.SH,隆基绿能,2024-09-30,0.100257,0.100257,False
13486,601012.SH,隆基绿能,2025-04-07,-0.100244,-0.100244,False
13182,601012.SH,隆基绿能,2023-12-28,0.100119,0.100119,False
6462,600030.SH,中信证券,2024-11-07,0.100110,0.100110,False
13366,601012.SH,隆基绿能,2024-10-09,-0.100108,-0.100108,False
4048,002475.SZ,立讯精密,2026-04-20,0.100084,0.100084,False
4193,002594.SZ,比亚迪,2021-03-08,-0.100060,-0.100060,False
6438,600030.SH,中信证券,2024-09-27,0.100048,0.100048,False
9857,600309.SH,万华化学,2021-09-13,0.100046,0.100046,False
13683,601012.SH,隆基绿能,2026-01-23,0.100043,0.100043,False


被剔除的跨期收益原始值（保留在 ret_raw，供敏感性比较）:


,stock_code,stock_name,date,skipped_days,ret_raw
1410,002352.SZ,顺丰控股,2021-02-10,3.0,0.100043
5792,600030.SH,中信证券,2022-01-27,6.0,-0.003300


### Step 3：Markdown结果解读

主键与排序检查通过：`(股票代码, 日期)` 重复数为0，10只股票均按日期升序排列。共同交易日历为1,384个交易日，正式样本期理论观测13,840条，实际13,831条，缺失9条，缺失率0.07%。

这9条缺失全部来自两只股票的停牌，不是接口返回缺口：顺丰控股缺2021-02-05、02-08、02-09共3日，因筹划收购嘉里物流股权于2月5日开市起临时停牌、2月10日复牌；中信证券缺2022-01-19至01-26共6日，因A股配股缴款期（1月19日至25日）全天停牌、1月26日网上清算继续停牌、1月27日开市复牌。判据是这些缺失日连续，且当日其余9只股票全部有行情，说明是这两只证券自身停止交易而非数据未返回；公开公告进一步印证了起止日期。9个缺失观测按规则不填0，直接不生成收益率。

按组内相邻后复权收盘价计算，共得到13,831个原始日收益，其中2个是跨期收益：顺丰控股2021-02-10跨越3个停牌日、原始值+10.0043%；中信证券2022-01-27跨越6个停牌日、原始值−0.3300%。这两个数字刻画的是停牌期加复牌当日的累计价格变化，不是单日收益。顺丰的+10.0043%尤其典型：若直接当单日收益会被读成涨停，实际是4个交易日的累计涨幅。因此主口径把它置为缺失，最终有效日收益13,829个，占理论样本13,840的99.92%；原始值保留在 `ret_raw` 列，后续必要时可做保留该观测的敏感性比较。

中信证券复牌日收益仅−0.33%，也值得单独说明：该日同时是配股除权日（配股价14.43元/股，每10股配售1.5股），后复权序列已包含配股的价格调整，因此价格序列连续，收益没有被除权的机械摊薄扭曲。

单位与合理性核对未发现技术错误。后复权收盘价全部为正，零成交量交易日为0；不复权收盘价区间11—2,601元，后复权价106.16—19,782.65元，成交量103万股—13.51亿股，成交额1.50—421.97亿元，历史总市值833.59—32,673.70亿元，各项数量级与A股实际情况相符。日收益绝对值超过10%的观测共22个，最大为隆基绿能2024-09-30的+10.0257%，超过11%的观测为0。10只股票均为主板（000/002/600/601），日涨跌幅限制为±10%，22个极端值全部紧贴限制边界，说明价格与复权序列没有出现单位错位或字段错配；个别略超10%来自收盘价两位小数的四舍五入，不是数据错误。

核心发现：

- 缺失9个观测全部由公司行为停牌造成，已用共同交易日历与公开公告双重印证，既不填0也不整只删除；
- 识别出2个跨期收益并在主口径中剔除，避免把停牌期累计变化误当单日收益，其中顺丰复牌日的+10.0043%是最容易被误读的一个；
- 主口径有效日收益13,829个，占理论样本的99.92%，样本损失极小且原因可解释；
- 单位、数量级与涨跌幅限制三项核对均通过，未发现需要修正的技术错误。

## 后续分析结构

后续各节继续严格采用“Step 1分析说明—Step 2代码与实际输出—Step 3结果解读”：

3. 不复权价格、累计收益和日收益图形  
4. 20日滚动年化波动率  
5. 描述统计与异常收益检查  
6. 日收益率Pearson相关性  
7. 等权组合与滞后总市值加权组合  
8. 总体结论、局限与AI使用说明
